# Difficulty-Based Sampling for Improving Mathematical Reasoning in Large Language Models

**Implementation notebook | GSM8K + Qwen2.5-0.5B-Instruct + GRPO**

This notebook organizes the implementation and recorded experiments into a clear sequence. It includes the pilot, fixed-sampling and adaptive-sampling experiments, a standard GRPO training experiment, an adaptive solver prototype, and saved-result analysis.

> **Execution note:** The full generation and GRPO training cells are compute-intensive. The experiment results recorded below were produced in earlier Colab runs and saved to Google Drive. Do not use **Runtime → Run all** unless you intend to rerun every experiment. Run the setup and sections individually. A GPU runtime is recommended for model generation and GRPO training.

**Research scope and honest limitation:** The adaptive sampling controller was evaluated separately. It was not directly integrated into the standard GRPO training loop. The later adaptive-data-allocation GRPO run was a small CPU prototype and should not be interpreted as proof of improvement.


## 1. Environment setup

In [ ]:
# Install the libraries used in the experiments (run once per fresh Colab runtime)
!pip install -q datasets transformers trl peft accelerate sentencepiece

import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. Model generation and GRPO training may be slow.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


## 2. Load and preprocess GSM8K

In [ ]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main")

print(dataset)

In [ ]:
import re

def extract_final_answer(answer):
    match = re.search(r"####\s*(.+)", answer)

    if match:
        return match.group(1).strip()

    return None


sample = dataset["train"][0]

print("Original Answer:")
print(sample["answer"])

print("\nExtracted Final Answer:")
print(extract_final_answer(sample["answer"]))

In [ ]:
def prepare_dataset(split):
    prepared_data = []

    for item in dataset[split]:
        final_answer = extract_final_answer(item["answer"])

        if final_answer is not None:
            prepared_data.append({
                "question": item["question"],
                "reasoning": item["answer"],
                "correct_answer": final_answer
            })

    return pd.DataFrame(prepared_data)


train_df = prepare_dataset("train")
test_df = prepare_dataset("test")

print("Training dataset shape:", train_df.shape)
print("Testing dataset shape:", test_df.shape)

print("\nPrepared training sample:")
print(train_df.iloc[0])

The GSM8K answer field contains a worked solution followed by the final answer marker `####`. The preprocessing function extracts that final numerical answer for evaluation.

## 3. Load Qwen2.5-0.5B-Instruct

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Model libraries imported successfully!")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

In [ ]:
# Device-aware model loading: supports GPU or CPU
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE
).to(DEVICE)
model.eval()

print("Qwen model loaded successfully")
print("Model device:", next(model.parameters()).device)


## 4. Answer extraction, evaluation and reward functions

In [ ]:
def extract_predicted_answer(response):

    # GSM8K format
    match = re.search(r"####\s*(-?\d[\d,]*(?:\.\d+)?)", response)

    if match:
        return match.group(1).replace(",", "")

    # Boxed answer format
    match = re.search(r"\\boxed\{(-?\d[\d,]*(?:\.\d+)?)\}", response)

    if match:
        return match.group(1).replace(",", "")

    # Explicit final answer format
    matches = re.findall(
        r"(?:final answer is|final answer:)\s*\**\s*(-?\d[\d,]*(?:\.\d+)?)",
        response,
        re.IGNORECASE
    )

    if matches:
        return matches[-1].replace(",", "")

    # Fallback: last numerical value
    numbers = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", response)

    if numbers:
        return numbers[-1].replace(",", "")

    return None


print("Improved answer extractor created successfully!")

In [ ]:
def evaluate_answer(predicted, correct):
    if predicted is None or correct is None:
        return False

    return float(predicted) == float(correct)


correct_answer = extract_final_answer(dataset["train"][0]["answer"])

is_correct = evaluate_answer(
    predicted_answer,
    correct_answer
)

print("Prediction:", predicted_answer)
print("Actual Answer:", correct_answer)
print("Evaluation:", "Correct" if is_correct else "Incorrect")

In [ ]:
def calculate_reward(predicted, correct):
    is_correct = evaluate_answer(predicted, correct)

    if is_correct:
        return 1.0
    else:
        return 0.0


reward = calculate_reward(
    predicted_answer,
    correct_answer
)

print("Predicted Answer:", predicted_answer)
print("Correct Answer:", correct_answer)
print("Reward:", reward)

The reward is binary: 1.0 for an exact numerical match and 0.0 otherwise. If the model response does not contain an extractable final answer, it is counted as incorrect.

## 5. Response generation

In [ ]:
def generate_response(question):

    messages = [
        {
            "role": "user",
            "content": (
                "Solve this mathematical problem step by step. "
                "Clearly provide the final numerical answer. "
                "End your response in this exact format: #### number\n\n"
                + question
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )


print("Updated generation function created successfully!")

## 6. Pilot experiment

Twenty training questions were sampled with a fixed random seed. The model generated three responses per question (60 generations total). The observed pilot accuracy was **76.67%**.

In [ ]:
import random
import pandas as pd
import torch

# Reproducible question selection
pilot_df = train_df.sample(
    n=20,
    random_state=42
).reset_index(drop=True)

pilot_results = []

print("Starting Pilot Experiment...")
print("Questions: 20")
print("Attempts per question: 3")
print("Total generations: 60")

for q_index, row in pilot_df.iterrows():

    question = row["question"]
    correct_answer = row["correct_answer"]

    for attempt in range(3):

        response = generate_response(question)

        predicted_answer = extract_predicted_answer(response)

        is_correct = evaluate_answer(
            predicted_answer,
            correct_answer
        )

        reward = calculate_reward(
            predicted_answer,
            correct_answer
        )

        pilot_results.append({
            "question_id": q_index + 1,
            "question": question,
            "attempt": attempt + 1,
            "response": response,
            "predicted_answer": predicted_answer,
            "correct_answer": correct_answer,
            "is_correct": is_correct,
            "reward": reward
        })

    print(f"Completed question {q_index + 1}/20")

# Store all experiment results
pilot_results_df = pd.DataFrame(pilot_results)

print("\nPILOT EXPERIMENT COMPLETED")
print("Total generations:", len(pilot_results_df))
print("Correct responses:", pilot_results_df["is_correct"].sum())
print("Incorrect responses:", (~pilot_results_df["is_correct"]).sum())

print("\nSample results:")
print(
    pilot_results_df[
        ["question_id", "attempt",
         "predicted_answer", "correct_answer",
         "is_correct", "reward"]
    ].head(10)
)

In [ ]:
difficulty_df = (
    pilot_results_df
    .groupby("question_id")
    .agg(
        total_attempts=("attempt", "count"),
        correct_attempts=("is_correct", "sum"),
        total_reward=("reward", "sum")
    )
    .reset_index()
)

difficulty_df["success_rate"] = (
    difficulty_df["correct_attempts"] /
    difficulty_df["total_attempts"]
)

difficulty_df["success_percentage"] = (
    difficulty_df["success_rate"] * 100
)

print("QUESTION-WISE PERFORMANCE")
print(difficulty_df.to_string(index=False))

print("\nOverall success rate:",
      round(difficulty_df["success_rate"].mean() * 100, 2), "%")

print("\nQuestions analyzed:", len(difficulty_df))

In [ ]:
def classify_difficulty(success_rate):

    if success_rate == 1.0:
        return "Easy"

    elif success_rate >= 0.66:
        return "Medium"

    elif success_rate > 0:
        return "Difficult"

    else:
        return "Very Difficult"


difficulty_df["difficulty"] = (
    difficulty_df["success_rate"].apply(classify_difficulty)
)

print("FINAL DIFFICULTY CLASSIFICATION\n")

print(
    difficulty_df[
        ["question_id", "success_percentage", "difficulty"]
    ].to_string(index=False)
)

print("\nDIFFICULTY SUMMARY")

print(
    difficulty_df["difficulty"].value_counts()
)

## 7. Fixed and difficulty-based adaptive sampling

The comparison used 20 questions and a total generation budget of 100. Fixed sampling generated five responses per question. Adaptive sampling began with three attempts per question and allocated the remaining budget according to the estimated difficulty.

In [ ]:
TOTAL_QUESTIONS = 20

FIXED_ATTEMPTS = 5
ADAPTIVE_INITIAL_ATTEMPTS = 3

TOTAL_BUDGET = TOTAL_QUESTIONS * FIXED_ATTEMPTS

MAX_EXTRA_ATTEMPTS = 5

DIFFICULTY_WEIGHTS = {
    "Easy": 0,
    "Medium": 1,
    "Difficult": 2,
    "Very Difficult": 3
}

print("Total sampling budget:", TOTAL_BUDGET)
print("Fixed sampling generations:", TOTAL_BUDGET)
print("Adaptive initial generations:",
      TOTAL_QUESTIONS * ADAPTIVE_INITIAL_ATTEMPTS)
print("Remaining adaptive budget:",
      TOTAL_BUDGET - TOTAL_QUESTIONS * ADAPTIVE_INITIAL_ATTEMPTS)

In [ ]:
adaptive_allocation_df = difficulty_df.copy()

adaptive_allocation_df["extra_attempts"] = 0

remaining_budget = (
    TOTAL_BUDGET -
    TOTAL_QUESTIONS * ADAPTIVE_INITIAL_ATTEMPTS
)

while remaining_budget > 0:

    eligible = adaptive_allocation_df[
        (adaptive_allocation_df["difficulty"] != "Easy") &
        (adaptive_allocation_df["extra_attempts"] < MAX_EXTRA_ATTEMPTS)
    ]

    if eligible.empty:
        raise ValueError("Not enough eligible questions for budget.")

    # Give priority according to difficulty and previous allocation
    priorities = eligible.apply(
        lambda row: DIFFICULTY_WEIGHTS[row["difficulty"]] /
        (row["extra_attempts"] + 1),
        axis=1
    )

    selected_index = priorities.idxmax()

    adaptive_allocation_df.loc[
        selected_index, "extra_attempts"
    ] += 1

    remaining_budget -= 1


adaptive_allocation_df["initial_attempts"] = (
    ADAPTIVE_INITIAL_ATTEMPTS
)

adaptive_allocation_df["total_attempts"] = (
    adaptive_allocation_df["initial_attempts"] +
    adaptive_allocation_df["extra_attempts"]
)

print("ADAPTIVE SAMPLING ALLOCATION")
print(
    adaptive_allocation_df[
        ["question_id", "difficulty",
         "initial_attempts", "extra_attempts",
         "total_attempts"]
    ].to_string(index=False)
)

print("\nTotal allocated generations:",
      adaptive_allocation_df["total_attempts"].sum())

In [ ]:
fixed_results = []

for q_index, row in pilot_df.iterrows():

    question = row["question"]
    correct_answer = row["correct_answer"]

    for attempt in range(FIXED_ATTEMPTS):

        response = generate_response(question)

        predicted_answer = extract_predicted_answer(response)

        is_correct = evaluate_answer(
            predicted_answer,
            correct_answer
        )

        reward = calculate_reward(
            predicted_answer,
            correct_answer
        )

        fixed_results.append({
            "question_id": q_index + 1,
            "attempt": attempt + 1,
            "predicted_answer": predicted_answer,
            "correct_answer": correct_answer,
            "is_correct": is_correct,
            "reward": reward
        })

    print(f"Completed question {q_index + 1}/20")

fixed_results_df = pd.DataFrame(fixed_results)

print("\nFIXED SAMPLING EXPERIMENT COMPLETED")
print("Total generations:", len(fixed_results_df))
print("Correct responses:", fixed_results_df["is_correct"].sum())
print("Incorrect responses:", (~fixed_results_df["is_correct"]).sum())

fixed_accuracy = (
    fixed_results_df["is_correct"].mean() * 100
)

print("Fixed Sampling Accuracy:", round(fixed_accuracy, 2), "%")

In [ ]:
adaptive_results = []

for _, allocation in adaptive_allocation_df.iterrows():

    question_id = allocation["question_id"]
    q_index = question_id - 1

    question = pilot_df.iloc[q_index]["question"]
    correct_answer = pilot_df.iloc[q_index]["correct_answer"]

    total_attempts = int(allocation["total_attempts"])

    for attempt in range(total_attempts):

        response = generate_response(question)

        predicted_answer = extract_predicted_answer(response)

        is_correct = evaluate_answer(
            predicted_answer,
            correct_answer
        )

        reward = calculate_reward(
            predicted_answer,
            correct_answer
        )

        adaptive_results.append({
            "question_id": question_id,
            "attempt": attempt + 1,
            "predicted_answer": predicted_answer,
            "correct_answer": correct_answer,
            "is_correct": is_correct,
            "reward": reward
        })

    print(f"Completed question {question_id}/20")

adaptive_results_df = pd.DataFrame(adaptive_results)

adaptive_accuracy = (
    adaptive_results_df["is_correct"].mean() * 100
)

print("\nADAPTIVE SAMPLING EXPERIMENT COMPLETED")
print("Total generations:", len(adaptive_results_df))
print("Correct responses:", adaptive_results_df["is_correct"].sum())
print("Incorrect responses:", (~adaptive_results_df["is_correct"]).sum())
print("Adaptive Sampling Accuracy:", round(adaptive_accuracy, 2), "%")

In [ ]:
# Add difficulty labels to both experiment results
fixed_analysis = fixed_results_df.merge(
    difficulty_df[["question_id", "difficulty"]],
    on="question_id"
)

adaptive_analysis = adaptive_results_df.merge(
    difficulty_df[["question_id", "difficulty"]],
    on="question_id"
)

# Calculate accuracy by difficulty
fixed_comparison = (
    fixed_analysis.groupby("difficulty")["is_correct"]
    .mean().mul(100).round(2)
)

adaptive_comparison = (
    adaptive_analysis.groupby("difficulty")["is_correct"]
    .mean().mul(100).round(2)
)

comparison_df = pd.DataFrame({
    "Fixed Sampling Accuracy (%)": fixed_comparison,
    "Adaptive Sampling Accuracy (%)": adaptive_comparison
}).fillna(0)

print("ACCURACY BY DIFFICULTY")
display(comparison_df)

# Calculate questions solved at least once
fixed_solved = (
    fixed_analysis.groupby("question_id")["is_correct"].any().sum()
)

adaptive_solved = (
    adaptive_analysis.groupby("question_id")["is_correct"].any().sum()
)

print("\nQUESTIONS SOLVED AT LEAST ONCE")
print("Fixed Sampling:", fixed_solved, "/ 20")
print("Adaptive Sampling:", adaptive_solved, "/ 20")

In [ ]:
fixed_question_summary = (
    fixed_analysis.groupby("question_id")
    .agg(
        fixed_attempts=("is_correct", "count"),
        fixed_correct=("is_correct", "sum")
    )
)

adaptive_question_summary = (
    adaptive_analysis.groupby("question_id")
    .agg(
        adaptive_attempts=("is_correct", "count"),
        adaptive_correct=("is_correct", "sum")
    )
)

question_comparison = fixed_question_summary.join(
    adaptive_question_summary
)

question_comparison["fixed_accuracy"] = (
    question_comparison["fixed_correct"] /
    question_comparison["fixed_attempts"] * 100
).round(2)

question_comparison["adaptive_accuracy"] = (
    question_comparison["adaptive_correct"] /
    question_comparison["adaptive_attempts"] * 100
).round(2)

display(question_comparison)

## 8. Standard GRPO training

The standard GRPO experiment used 100 training questions selected after excluding the 20 pilot questions. LoRA was used to reduce the number of trainable parameters. The recorded run completed 20 training steps. This is a separate standard GRPO experiment; it did not use the adaptive controller inside its rollout loop.

In [ ]:
from datasets import Dataset

# Exclude the 20 pilot experiment questions
pilot_questions = set(pilot_df["question"])

grpo_train_df = train_df[
    ~train_df["question"].isin(pilot_questions)
].sample(n=100, random_state=42).reset_index(drop=True)

# Prepare prompts for GRPO
def create_grpo_prompt(question):
    return (
        "Solve this mathematical problem step by step. "
        "Clearly provide the final numerical answer. "
        "End your response in this exact format: #### number\n\n"
        + question
    )

grpo_train_df["prompt"] = grpo_train_df["question"].apply(
    create_grpo_prompt
)

# Convert to Hugging Face Dataset
grpo_train_dataset = Dataset.from_pandas(
    grpo_train_df[["prompt", "correct_answer"]],
    preserve_index=False
)

print("GRPO training dataset prepared successfully!")
print("Number of training questions:", len(grpo_train_dataset))
print("\nDataset columns:", grpo_train_dataset.column_names)
print("\nSample prompt:")
print(grpo_train_dataset[0]["prompt"])
print("\nCorrect answer:", grpo_train_dataset[0]["correct_answer"])

In [ ]:
def gsm8k_accuracy_reward(completions, correct_answer, **kwargs):

    rewards = []

    for completion, answer in zip(completions, correct_answer):

        # Handle text or chat-style model responses
        if isinstance(completion, list):
            response = completion[-1]["content"]
        else:
            response = completion

        # Extract the predicted numerical answer
        predicted = extract_predicted_answer(response)

        # Compare predicted and actual answers
        is_correct = evaluate_answer(predicted, answer)

        if is_correct:
            rewards.append(1.0)
        else:
            rewards.append(0.0)

    return rewards


print("GRPO reward function created successfully!")

# Test the reward function
test_completions = [
    "The answer is #### 80",
    "The answer is #### 50"
]

test_answers = ["80", "80"]

print(
    "Test rewards:",
    gsm8k_accuracy_reward(test_completions, test_answers)
)

In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

print("LoRA configuration created successfully!")
print("Rank:", lora_config.r)
print("LoRA Alpha:", lora_config.lora_alpha)
print("LoRA Dropout:", lora_config.lora_dropout)
print("Target modules:", lora_config.target_modules)

In [ ]:
from trl import GRPOConfig

grpo_config = GRPOConfig(
    output_dir="./qwen_grpo_results",

    # Training settings
    max_steps=20,
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,

    # GRPO settings
    num_generations=4,
    max_completion_length=256,
    beta=0.04,

    # GPU memory settings
    fp16=(DEVICE == "cuda"),
    bf16=False,
    gradient_checkpointing=True,
    optim="adamw_torch",

    # Logging and saving
    logging_steps=1,
    save_strategy="no",
    report_to="none",

    seed=42
)

print("GRPO training configuration created!")
print("Maximum training steps:", grpo_config.max_steps)
print("Learning rate:", grpo_config.learning_rate)
print("Generations per prompt:", grpo_config.num_generations)
print("Completion length:", grpo_config.max_completion_length)
print("Batch size:", grpo_config.per_device_train_batch_size)
print("FP16:", grpo_config.fp16)

In [ ]:
# Convert each prompt into the Qwen chat-message format
def convert_to_chat_format(example):
    return {
        "prompt": [
            {
                "role": "user",
                "content": example["prompt"]
            }
        ]
    }

grpo_train_dataset = grpo_train_dataset.map(
    convert_to_chat_format
)

# Configure padding token if required
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Chat format conversion completed!")
print("\nSample training prompt:")
print(grpo_train_dataset[0]["prompt"])
print("\nCorrect answer:", grpo_train_dataset[0]["correct_answer"])

In [ ]:
# Initialize and run GRPO training (compute-intensive; run only when intentionally reproducing training)
model.config.use_cache = False

grpo_trainer = GRPOTrainer(
    model=model,
    reward_funcs=gsm8k_accuracy_reward,
    args=grpo_config,
    train_dataset=grpo_train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config
)

grpo_train_result = grpo_trainer.train()
print("GRPO training completed")
print(grpo_train_result.metrics)


In [ ]:
# Save the trained LoRA adapter
SAVE_PATH = "./qwen_grpo_trained"

grpo_trainer.model.save_pretrained(SAVE_PATH)

# Save tokenizer
tokenizer.save_pretrained(SAVE_PATH)

print("Trained GRPO model saved successfully!")
print("Save location:", SAVE_PATH)

## 9. Model evaluation and result visualization

In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel
import torch

# Load a fresh original Qwen model
base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE
).to(DEVICE)

# Attach the trained GRPO adapter
trained_model = PeftModel.from_pretrained(
    base_model_eval,
    SAVE_PATH
)

trained_model.eval()

print("Clean base model loaded successfully!")
print("Trained GRPO adapter attached successfully!")
print("Device:", next(trained_model.parameters()).device)
print("Active adapters:", trained_model.active_adapters)

In [ ]:
def generate_trained_response(question):

    messages = [
        {
            "role": "user",
            "content": (
                "Solve this mathematical problem step by step. "
                "Clearly provide the final numerical answer. "
                "End your response in this exact format: #### number\n\n"
                + question
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = trained_model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )


test_question = test_df.iloc[0]["question"]

trained_response = generate_trained_response(test_question)

print("QUESTION:")
print(test_question)

print("\nTRAINED MODEL RESPONSE:")
print(trained_response)

print("\nPREDICTED ANSWER:")
print(extract_predicted_answer(trained_response))

print("\nCORRECT ANSWER:")
print(test_df.iloc[0]["correct_answer"])

In [ ]:
def evaluate_answer(predicted, correct):

    if predicted is None or correct is None:
        return False

    predicted = str(predicted).replace(",", "").strip()
    correct = str(correct).replace(",", "").strip()

    try:
        return float(predicted) == float(correct)
    except ValueError:
        return False


print("Updated answer evaluator successfully!")

# Test
print("Test 1:", evaluate_answer("65,960", "65,960"))
print("Test 2:", evaluate_answer("72", "50"))

In [ ]:
import pandas as pd
import torch

# Select 100 unseen test questions
evaluation_df = test_df.sample(
    n=100,
    random_state=42
).reset_index(drop=True)

def generate_with_model(active_model, question):
    messages = [
        {
            "role": "user",
            "content": (
                "Solve this mathematical problem step by step. "
                "Clearly provide the final numerical answer. "
                "End your response in this exact format: #### number\n\n"
                + question
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = active_model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )


evaluation_results = []

for i, row in evaluation_df.iterrows():

    question = row["question"]
    correct_answer = row["correct_answer"]

    # Original model: disable LoRA adapter
    with trained_model.disable_adapter():
        original_response = generate_with_model(
            trained_model, question
        )

    original_prediction = extract_predicted_answer(
        original_response
    )

    # Trained model: use LoRA adapter
    trained_response = generate_with_model(
        trained_model, question
    )

    trained_prediction = extract_predicted_answer(
        trained_response
    )

    evaluation_results.append({
        "question_id": i + 1,
        "correct_answer": correct_answer,
        "original_prediction": original_prediction,
        "trained_prediction": trained_prediction,
        "original_correct": evaluate_answer(
            original_prediction, correct_answer
        ),
        "trained_correct": evaluate_answer(
            trained_prediction, correct_answer
        )
    })

    print(f"Completed {i + 1}/100")

evaluation_results_df = pd.DataFrame(evaluation_results)

original_accuracy = (
    evaluation_results_df["original_correct"].mean() * 100
)

trained_accuracy = (
    evaluation_results_df["trained_correct"].mean() * 100
)

print("\nFINAL MODEL EVALUATION")
print("Total test questions:", len(evaluation_results_df))
print("Original model accuracy:", round(original_accuracy, 2), "%")
print("GRPO trained model accuracy:", round(trained_accuracy, 2), "%")
print(
    "Accuracy improvement:",
    round(trained_accuracy - original_accuracy, 2),
    "percentage points"
)

In [ ]:
import os
import json

# Create project results folder
RESULTS_PATH = "./research_project_results"
os.makedirs(RESULTS_PATH, exist_ok=True)

# Save experiment tables
pilot_results_df.to_csv(
    f"{RESULTS_PATH}/pilot_results.csv", index=False
)

difficulty_df.to_csv(
    f"{RESULTS_PATH}/difficulty_analysis.csv", index=False
)

adaptive_allocation_df.to_csv(
    f"{RESULTS_PATH}/adaptive_allocation.csv", index=False
)

fixed_results_df.to_csv(
    f"{RESULTS_PATH}/fixed_sampling_results.csv", index=False
)

adaptive_results_df.to_csv(
    f"{RESULTS_PATH}/adaptive_sampling_results.csv", index=False
)

question_comparison.to_csv(
    f"{RESULTS_PATH}/question_comparison.csv"
)

evaluation_results_df.to_csv(
    f"{RESULTS_PATH}/grpo_evaluation_results.csv", index=False
)

# Save evaluation metrics
final_metrics = {
    "fixed_sampling_accuracy": float(fixed_accuracy),
    "adaptive_sampling_accuracy": float(adaptive_accuracy),
    "original_model_accuracy": float(original_accuracy),
    "grpo_trained_model_accuracy": float(trained_accuracy),
    "training_steps": 20,
    "training_questions": 100,
    "evaluation_questions": 100
}

with open(
    f"{RESULTS_PATH}/final_metrics.json", "w"
) as file:
    json.dump(final_metrics, file, indent=4)

print("ALL EXPERIMENT RESULTS SAVED SUCCESSFULLY!")
print("\nSaved files:")

for filename in os.listdir(RESULTS_PATH):
    print("-", filename)

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

methods = [
    "Fixed Sampling",
    "Adaptive Sampling",
    "Original Model",
    "GRPO-Trained Model"
]

accuracies = [79, 54, 42, 41]

plt.figure(figsize=(9, 5))

bars = plt.bar(methods, accuracies)

plt.title("Accuracy Comparison of Sampling and Training Methods")
plt.xlabel("Methods")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.xticks(rotation=15)

for bar, accuracy in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{accuracy}%",
        ha="center"
    )

plt.tight_layout()
plt.savefig(
    "./research_project_results/accuracy_comparison.png",
    dpi=300
)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np

difficulty_levels = ["Easy", "Medium", "Difficult"]

fixed_accuracy = [90.91, 75.00, 56.00]
adaptive_accuracy = [78.79, 55.56, 32.50]

x = np.arange(len(difficulty_levels))
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(x - width/2, fixed_accuracy, width, label="Fixed Sampling")
plt.bar(x + width/2, adaptive_accuracy, width, label="Adaptive Sampling")

plt.xlabel("Difficulty Level")
plt.ylabel("Accuracy (%)")
plt.title("Fixed vs Adaptive Sampling by Difficulty")
plt.xticks(x, difficulty_levels)
plt.ylim(0, 100)
plt.legend()

plt.tight_layout()
plt.savefig(
    "./research_project_results/difficulty_comparison.png",
    dpi=300
)
plt.show()

In [ ]:
summary = """
RESEARCH PROJECT: DIFFICULTY-BASED SAMPLING
FOR IMPROVING MATHEMATICAL REASONING IN LLMS

1. PILOT EXPERIMENT
Questions: 20
Attempts per question: 3
Total generations: 60
Accuracy: 76.67%

2. FIXED SAMPLING
Questions: 20
Attempts per question: 5
Total generations: 100
Accuracy: 79%

3. ADAPTIVE SAMPLING
Questions: 20
Initial attempts: 3
Total generation budget: 100
Accuracy: 54%

4. GRPO TRAINING
Base model: Qwen2.5-0.5B-Instruct
Training questions: 100
Training steps: 20
Original model accuracy: 42%
GRPO-trained model accuracy: 41%

5. DIFFICULTY-WISE COMPARISON
Easy:
Fixed: 90.91%
Adaptive: 78.79%

Medium:
Fixed: 75%
Adaptive: 55.56%

Difficult:
Fixed: 56%
Adaptive: 32.50%

6. MAIN FINDING
Fixed sampling achieved the highest accuracy.
Adaptive sampling did not outperform fixed sampling.
GRPO training showed a 1 percentage-point decrease
compared with the original model.

7. IMPORTANT LIMITATION
The adaptive sampling controller was evaluated separately.
It was not directly integrated into the GRPO training loop.
The model evaluation used a sample of 100 held-out test questions.
"""

with open(
    "./research_project_results/experiment_summary.txt",
    "w"
) as file:
    file.write(summary)

print("Experiment summary saved successfully!")

## 10. Difficulty-based adaptive solver prototype

This standalone inference-time prototype generates three initial responses, estimates difficulty from answer agreement, and assigns additional attempts. Its three-question demonstration is illustrative only; it is not a statistically reliable evaluation.

In [ ]:
# ============================================================
# DIFFICULTY-BASED ADAPTIVE MATHEMATICAL REASONING SYSTEM
# Model: Qwen2.5-0.5B-Instruct
# ============================================================

import re
import os
import json
import torch
import pandas as pd
from collections import Counter

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

DEVICE = next(model.parameters()).device
INITIAL_ATTEMPTS = 3

print("Model: Qwen2.5-0.5B-Instruct")
print("Device:", DEVICE)


# ------------------------------------------------------------
# 2. ANSWER EXTRACTION
# ------------------------------------------------------------

def extract_final_answer(response):

    if not response:
        return None

    text = response.replace(",", "").strip()

    # Priority 1: GSM8K answer format
    matches = re.findall(
        r"####\s*([-+]?\d+(?:\.\d+)?)",
        text
    )

    if matches:
        return matches[-1].rstrip(".")

    # Priority 2: Explicit final answer
    patterns = [
        r"(?:final answer|answer)\s*(?:is|:|=)\s*"
        r"(?:Rs\.?|₹|\$)?\s*([-+]?\d+(?:\.\d+)?)",

        r"(?:therefore|thus|hence)[^.\n]*?"
        r"(?:is|equals|=)\s*(?:Rs\.?|₹|\$)?\s*"
        r"([-+]?\d+(?:\.\d+)?)",

        r"(?:cost|total)\s*(?:is|equals|=)\s*"
        r"(?:Rs\.?|₹|\$)?\s*([-+]?\d+(?:\.\d+)?)"
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].rstrip(".")

    # Priority 3: Numeric answer on the final line only
    last_line = text.splitlines()[-1].strip()

    match = re.search(
        r"^(?:the answer is\s*)?"
        r"(?:Rs\.?|₹|\$)?\s*([-+]?\d+(?:\.\d+)?)"
        r"\s*[.!]?$",
        last_line,
        re.IGNORECASE
    )

    if match:
        return match.group(1).rstrip(".")

    return None


# ------------------------------------------------------------
# 3. GENERATE ANSWER USING QWEN
# ------------------------------------------------------------

def generate_qwen_answer(question, temperature=0.7):

    messages = [
        {
            "role": "user",
            "content": (
                "Solve this mathematical problem carefully. "
                "Show the calculation step by step. "
                "Do not use numbers from the question as the final answer "
                "unless they are actually the result. "
                "End with the final numerical answer on a separate line "
                "in this exact format: #### number\n\n"
                + question
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)

    generation_args = {
        "max_new_tokens": 384,
        "pad_token_id": tokenizer.eos_token_id
    }

    if temperature > 0:
        generation_args.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": 0.9
        })
    else:
        generation_args["do_sample"] = False

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **generation_args
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    answer = extract_final_answer(response)

    return answer, response


# ------------------------------------------------------------
# 4. DIFFICULTY ESTIMATION
# ------------------------------------------------------------

def estimate_difficulty(answers):

    valid_answers = [
        str(a).rstrip(".")
        for a in answers
        if a is not None
    ]

    if not valid_answers:
        return "Very Difficult", 0.0

    counts = Counter(valid_answers)

    agreement = counts.most_common(1)[0][1] / len(answers)

    if agreement == 1.0:
        difficulty = "Easy"
    elif agreement >= 0.66:
        difficulty = "Medium"
    elif agreement > 0:
        difficulty = "Difficult"
    else:
        difficulty = "Very Difficult"

    return difficulty, agreement


# ------------------------------------------------------------
# 5. ADAPTIVE SAMPLING CONTROLLER
# ------------------------------------------------------------

def adaptive_sampling_controller(difficulty):

    allocation = {
        "Easy": 0,
        "Medium": 2,
        "Difficult": 3,
        "Very Difficult": 4
    }

    return allocation[difficulty]


# ------------------------------------------------------------
# 6. CHECK ANSWER
# ------------------------------------------------------------

def check_answer(predicted, expected):

    if predicted is None or expected is None:
        return False

    try:
        predicted_number = float(
            str(predicted).replace(",", "").strip()
        )

        expected_number = float(
            str(expected).replace(",", "").strip()
        )

        return abs(predicted_number - expected_number) < 1e-6

    except (ValueError, TypeError):
        return False


# ------------------------------------------------------------
# 7. COMPLETE ADAPTIVE MATHEMATICAL SOLVER
# ------------------------------------------------------------

def adaptive_math_solver(question, expected_answer=None):

    print("\nGenerating initial responses...")

    answers = []

    # Initial attempts
    for i in range(INITIAL_ATTEMPTS):

        answer, raw_response = generate_qwen_answer(
            question,
            temperature=0.7
        )

        answers.append(answer)

        print(
            f"Initial attempt {i+1}: "
            f"{answer if answer is not None else 'Answer not extracted'}"
        )

    # Estimate difficulty
    difficulty, agreement = estimate_difficulty(answers)

    print("\nEstimated difficulty:", difficulty)
    print("Answer consistency:", round(agreement * 100, 2), "%")

    # Allocate additional attempts
    extra_attempts = adaptive_sampling_controller(difficulty)

    print("Additional attempts:", extra_attempts)

    for i in range(extra_attempts):

        answer, raw_response = generate_qwen_answer(
            question,
            temperature=0.7
        )

        answers.append(answer)

        print(
            f"Additional attempt {i+1}: "
            f"{answer if answer is not None else 'Answer not extracted'}"
        )

    # Majority voting
    valid_answers = [
        str(a).rstrip(".")
        for a in answers
        if a is not None
    ]

    if valid_answers:
        predicted_answer = Counter(
            valid_answers
        ).most_common(1)[0][0]
    else:
        predicted_answer = None

    final_agreement = (
        Counter(valid_answers).most_common(1)[0][1] / len(answers)
        if valid_answers else 0.0
    )

    correct = (
        check_answer(predicted_answer, expected_answer)
        if expected_answer is not None else None
    )

    print("\n" + "=" * 55)
    print("FINAL ADAPTIVE MATHEMATICAL REASONING RESULT")
    print("=" * 55)

    print("Question:", question)
    print("Difficulty:", difficulty)
    print("Initial attempts:", INITIAL_ATTEMPTS)
    print("Additional attempts:", extra_attempts)
    print("Total attempts:", len(answers))
    print("Predicted answer:", predicted_answer)
    print("Expected answer:", expected_answer)
    print("Answer agreement:", round(final_agreement * 100, 2), "%")

    if correct is not None:
        print("Correct:", correct)

    return {
        "question": question,
        "difficulty": difficulty,
        "initial_attempts": INITIAL_ATTEMPTS,
        "additional_attempts": extra_attempts,
        "total_attempts": len(answers),
        "predicted_answer": predicted_answer,
        "expected_answer": expected_answer,
        "agreement": round(final_agreement * 100, 2),
        "correct": correct,
        "all_answers": answers
    }


# ------------------------------------------------------------
# 8. TEST ON THREE MATHEMATICAL QUESTIONS
# ------------------------------------------------------------

test_questions = [
    {
        "question": (
            "Ravi has 12 apples and buys 8 more apples. "
            "How many apples does he have in total?"
        ),
        "expected_answer": "20"
    },
    {
        "question": (
            "A notebook costs Rs. 25. "
            "What is the cost of 6 notebooks?"
        ),
        "expected_answer": "150"
    },
    {
        "question": (
            "A shopkeeper buys 5 bags for Rs. 750. "
            "What is the cost of 8 bags at the same rate?"
        ),
        "expected_answer": "1200"
    }
]

test_results = []

for index, item in enumerate(test_questions, 1):

    print("\n\n" + "#" * 60)
    print("TEST QUESTION", index)
    print("#" * 60)

    result = adaptive_math_solver(
        item["question"],
        item["expected_answer"]
    )

    test_results.append(result)


# ------------------------------------------------------------
# 9. DISPLAY EXPERIMENT SUMMARY
# ------------------------------------------------------------

results_df = pd.DataFrame([
    {
        "Question": r["question"],
        "Difficulty": r["difficulty"],
        "Initial Attempts": r["initial_attempts"],
        "Additional Attempts": r["additional_attempts"],
        "Total Attempts": r["total_attempts"],
        "Predicted Answer": r["predicted_answer"],
        "Expected Answer": r["expected_answer"],
        "Agreement (%)": r["agreement"],
        "Correct": r["correct"]
    }
    for r in test_results
])

print("\n\nFINAL TEST SUMMARY")
display(results_df)

valid_results = [
    r for r in test_results
    if r["correct"] is not None
]

if valid_results:
    accuracy = (
        sum(r["correct"] for r in valid_results)
        / len(valid_results)
    ) * 100

    print("Test accuracy:", round(accuracy, 2), "%")
else:
    accuracy = None
    print("Accuracy unavailable.")


# ------------------------------------------------------------
# 10. SAVE RESULTS TO GOOGLE DRIVE
# ------------------------------------------------------------

SAVE_FOLDER = "/content/drive/MyDrive/research_project_results"

os.makedirs(SAVE_FOLDER, exist_ok=True)

results_df.to_csv(
    os.path.join(SAVE_FOLDER, "final_adaptive_test_results.csv"),
    index=False
)

summary = {
    "model": "Qwen2.5-0.5B-Instruct",
    "device": str(DEVICE),
    "number_of_test_questions": len(test_results),
    "test_accuracy_percent": accuracy,
    "results": test_results
}

with open(
    os.path.join(SAVE_FOLDER, "final_adaptive_test_summary.json"),
    "w"
) as file:
    json.dump(summary, file, indent=4)

print("\nResults saved to Google Drive:")
print(SAVE_FOLDER)


# ------------------------------------------------------------
# 11. SIMPLE INTERACTIVE INTERFACE
# ------------------------------------------------------------

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    question_box = widgets.Textarea(
        placeholder="Enter your mathematical question here...",
        description="Question:",
        layout=widgets.Layout(width="90%", height="100px")
    )

    solve_button = widgets.Button(
        description="Solve Question",
        button_style="success",
        icon="check"
    )

    output_area = widgets.Output()

    def solve_button_clicked(button):

        with output_area:
            clear_output()

            question = question_box.value.strip()

            if not question:
                print("Please enter a mathematical question.")
                return

            adaptive_math_solver(question)

    solve_button.on_click(solve_button_clicked)

    print("\nINTERACTIVE MATHEMATICAL REASONING SYSTEM")

    display(question_box)
    display(solve_button)
    display(output_area)

except ImportError:

    print(
        "\nipywidgets is unavailable. "
        "The adaptive solver can still be used directly."
    )

print("\nIMPLEMENTATION CELL FINISHED.")


In [ ]:
# TEST THE ADAPTIVE MODEL ON 3 MATH QUESTIONS

test_questions = [
    {
        "question": "Ravi has 12 apples and buys 8 more apples. How many apples does he have in total?",
        "expected_answer": "20"
    },
    {
        "question": "A notebook costs Rs. 25. What is the cost of 6 notebooks?",
        "expected_answer": "150"
    },
    {
        "question": "A shopkeeper buys 5 bags for Rs. 750. What is the cost of 8 bags at the same rate?",
        "expected_answer": "1200"
    }
]

test_results = []

for i, item in enumerate(test_questions, 1):
    print("\n" + "=" * 55)
    print(f"TEST QUESTION {i}")
    print("=" * 55)

    result = adaptive_math_solver(
        item["question"],
        item["expected_answer"]
    )

    test_results.append(result)

print("\nALL THREE TESTS FINISHED.")


## 11. Adaptive-data-allocation GRPO prototype: scope and limitation

A later small CPU prototype constructed an adaptive training dataset and completed a very short GRPO run (2 steps). It is retained as an exploratory prototype, not as the main GRPO result. The main project limitation remains: the sampling controller was not integrated into the original GRPO rollout process. The small prototype did not establish an accuracy gain.
The saved adapter and evaluation CSV are retained in the Google Drive results folder. This notebook keeps the primary reproducible training path separate so the two experiments are not confused.


## 12. Recorded experimental results

### Sampling comparison

| Experiment | Questions | Generations | Accuracy |
|---|---:|---:|---:|
| Pilot | 20 | 60 | 76.67% |
| Fixed sampling | 20 | 100 | 79% |
| Adaptive sampling | 20 | 100 | 54% |

### GRPO comparison

| Model | Evaluation questions | Accuracy |
|---|---:|---:|
| Original Qwen2.5-0.5B-Instruct | 100 | 42% |
| GRPO-trained model | 100 | 41% |

### Accuracy by estimated difficulty

| Difficulty | Fixed sampling | Adaptive sampling |
|---|---:|---:|
| Easy | 90.91% | 78.79% |
| Medium | 75% | 55.56% |
| Difficult | 56% | 32.50% |


In [ ]:
# Load the previously saved result files from Google Drive.
# Mount Drive first if this is a fresh Colab runtime.
from google.colab import drive
drive.mount('/content/drive')

RESULTS_PATH = "/content/drive/MyDrive/research_project_results"
print("Saved result files:")
for filename in sorted(os.listdir(RESULTS_PATH)):
    print("-", filename)

with open(os.path.join(RESULTS_PATH, "final_metrics.json"), "r") as f:
    final_metrics = json.load(f)
print("\nMain experiment metrics:")
print(json.dumps(final_metrics, indent=4))

with open(os.path.join(RESULTS_PATH, "experiment_summary.txt"), "r") as f:
    print("\nExperiment summary:\n")
    print(f.read())


In [ ]:
from IPython.display import Image, display

accuracy_graph = os.path.join(RESULTS_PATH, "accuracy_comparison.png")
difficulty_graph = os.path.join(RESULTS_PATH, "difficulty_comparison.png")

if os.path.exists(accuracy_graph):
    display(Image(filename=accuracy_graph))
if os.path.exists(difficulty_graph):
    display(Image(filename=difficulty_graph))


## 13. Findings, limitations and conclusion

### Findings
- Fixed sampling achieved the highest observed accuracy (79%) under the 100-generation comparison budget.
- Adaptive sampling achieved 54% and did not outperform fixed sampling in this experiment.
- Standard GRPO training changed the measured accuracy from 42% to 41% on the 100-question evaluation sample.
- Difficulty-wise results also favored fixed sampling across the reported easy, medium and difficult groups.

### Limitations
1. The adaptive controller was evaluated separately and was not directly integrated into the standard GRPO rollout/training loop.
2. The reported model evaluation used a sample of 100 held-out GSM8K test questions.
3. The adaptive three-question demonstration is small and should not be used to claim general model accuracy.
4. Some generations did not contain an extractable final answer; these were treated as incorrect.
5. The adaptive-data-allocation GRPO run was a short CPU prototype and did not establish a performance improvement.

### Conclusion
Under the tested setup and generation budget, fixed sampling performed better than the implemented adaptive sampling strategy. The standard GRPO run did not improve the measured held-out accuracy. The results suggest that the current difficulty estimator and allocation strategy require further refinement and more extensive validation.


## 14. Reproducibility checklist

- [ ] Confirm Colab runtime and install required libraries.
- [ ] Mount Google Drive and verify `research_project_results` exists.
- [ ] Confirm GSM8K and Qwen2.5-0.5B-Instruct load correctly.
- [ ] Review the sampling and GRPO code before intentionally rerunning expensive cells.
- [ ] Confirm both saved graphs and all result CSV/JSON files are present.
- [ ] Keep the reported limitations with the results; do not describe the adaptive controller as integrated into the main GRPO loop.
